# Einführung in Python für die Computational Social Science (CSS)

## Jonas Volle
Wissenschaftlicher Mitarbeiter  
Chair of Methodology and Empirical Social Research  
Otto-von-Guericke-Universität

[jonas.volle@ovgu.de](mailto:jonas.volle@ovgu.de)

**Sprechstunde**: individuell nach vorheriger Anmeldung per [Mail](mailto:jonas.volle@ovgu.de)

Donnerstag, 26.06.2025

**Quelle:** Ich orientiere mich für diese Sitzung in Teilen am Kapitel 7 aus dem Buch:  

McLevey, John. 2021. Doing Computational Social Science: A Practical Introduction. 1st ed. Thousand Oaks: SAGE Publications.

und der Introduction to Computational Social Science methods with Python von GESIS unter: https://github.com/gesiscss/css_methods_python 

# Session 5: Textanalysen 

## Import der Textdaten

In [ ]:
import pandas as pd
import cred
import requests
import pprint as pp
import time
from bs4 import BeautifulSoup

GUARDIAN_KEY = cred.GUARDIAN_KEY

In [ ]:
# API Endpoint
API_ENDPOINT = 'http://content.guardianapis.com/search'

# API Parameter
PARAMS = {
    'api-key': GUARDIAN_KEY,
    'from-date': '2024-01-01',
    'to-date': '2025-04-30',
    'lang': 'en',
    'production-office': 'uk',
    'q': 'ukraine OR russia',
    'show-fields': 'wordcount,body,byline',
    'page-size': 50
} 


In [ ]:
# GET request

response = requests.get(API_ENDPOINT, params=PARAMS) 
response_dict = response.json()['response']

In [ ]:
response_dict['total']

In [ ]:
response_dict['pages']

In [ ]:
all_results = []
cur_page = 1
total_pages = 1

while (cur_page <= total_pages) and (cur_page < 200):

    # Make API request
    PARAMS['page'] = cur_page
    response = requests.get(API_ENDPOINT, params=PARAMS) 
    response_dict = response.json()['response']

    # update total pages
    total_pages = response_dict['pages']

    print(f"page: {cur_page} of {total_pages}")

    # update cur page
    cur_page += 1

    # append result
    all_results += (response_dict['results'])

    # sleep
    time.sleep(1)

In [ ]:
all_results_df = pd.json_normalize(all_results)

In [ ]:
all_results_df['text'] = [BeautifulSoup(i, "html.parser").text for i in all_results_df['fields.body']]

In [ ]:
# date
all_results_df['article_date'] = pd.to_datetime(all_results_df.webPublicationDate)

# rename columns
all_results_df = all_results_df.rename(columns={'webTitle':'article_title',
                                               'webUrl':'article_url',
                                               'fields.byline': 'article_author',
                                               'sectionName': 'section_name',
                                               'pillarName': 'pillar_name'})

# filter columns
all_results_df_f = all_results_df[['id', 'article_date', 'section_name', 'pillar_name',
                                   'article_title', 'article_url', 
                                   'article_author', 'text']].copy()

In [ ]:
all_results_df_f.info()

In [ ]:
all_results_df_f.head()

In [ ]:
# # export full data
# all_results_df_f.to_csv('data/guardian_ukraine_russia_textdata.csv', index= False)

In [ ]:
# export sampled data

# all_results_df_sample_100 = all_results_df_f.sample(100, random_state=1234)
# all_results_df_sample_100.to_csv('data/guardian_ukraine_russia_textdata_sample_100.csv',
#                                  index= False)

# all_results_df_sample_200 = all_results_df_f.sample(200, random_state=1234)
# all_results_df_sample_200.to_csv('data/guardian_ukraine_russia_textdata_sample_200.csv',
#                                  index= False)

# all_results_df_sample_500 = all_results_df_f.sample(500, random_state=1234)
# all_results_df_sample_500.to_csv('data/guardian_ukraine_russia_textdata_sample_500.csv',
#                                  index= False)

# all_results_df_sample_1000 = all_results_df_f.sample(1000, random_state=1234)
# all_results_df_sample_1000.to_csv('data/guardian_ukraine_russia_textdata_sample_1000.csv',
#                                  index= False)

## Natural Language Processing

Wir erstellen ein neues anaconda environment für Textanalysen:

`conda create -c conda-forge --name python_nlp python=3.9 ipykernel pandas numpy requests beautifulsoup4 nltk matplotlib spacy gensim pyldavis`

und laden folgende Sprachmodelle in in diesem environment herunter:

`python -m nltk.downloader popular`

`python -m spacy download en_core_web_sm`

Zunächst werden die Textdaten importiert.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/guardian_ukraine_russia_textdata_sample_500.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

### Tokenization

Für die Analyse werden die Texte in Analyseeinheiten zerteilt.

Die durch Leerzeichen und Interpunktion getrennten Wörter eines Textdokuments werden als Token bezeichnet.

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
example_text = df.text[0]
print(example_text)

In [ ]:
word_tokenize(example_text)

Wir können auch alle tokens in eine List packen, um die häufigsten token zu zählen.

In [ ]:
from collections import Counter

# Alle tokens in einer Liste
tokens = []
for text in df.text:
    doc = word_tokenize(text)
    for token in doc:
        tokens.append(token)

vocabulary = Counter(tokens)

In [ ]:
vocabulary.most_common(20)

<div class='alert alert-block alert-warning'>

### Zipfsche Gesetz

In vielen natürlichen Texten (wie Büchern, Artikeln, Gesprächen) kommt das häufigste Wort etwa doppelt so oft vor wie das zweithäufigste, dreimal so oft wie das dritthäufigste usw.

</div>

In [ ]:
import matplotlib.pyplot as plt

# Zipf Verteilung

# Gesamtanzahl der Tokens
total_tokens = sum(vocabulary.values())

# Wörter nach Häufigkeit sortieren
sorted_vocab = vocabulary.most_common(100)

# Rang (x-Achse) und relative Häufigkeit (y-Achse)
ranks = range(1, len(sorted_vocab) + 1)
relative_freqs = [count / total_tokens for _, count in sorted_vocab]
abs_freqs = [count for _, count in sorted_vocab]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(ranks, relative_freqs)
# plt.xscale('log')
# plt.yscale('log')
plt.xlabel('Rang des Wortes')
plt.ylabel('Relative Häufigkeit')
plt.title('Zipf-Plot der Wortverteilung')
plt.grid(True)
plt.show()

### Stemming

Beim Stemming werden die Suffixe von Wörtern entfernt, um eine vereinfachte Form des Wortes zu erhalten.

running, runner, run -> run

Ein weit verbreiteter Stemming Algorithmus ist der von Porter.

In [ ]:
from nltk.stem import PorterStemmer

In [ ]:
stemmer = PorterStemmer()

In [ ]:
example_df = pd.DataFrame({'token': word_tokenize(example_text)})
example_df.head()

In [ ]:
example_df['stem'] = [stemmer.stem(token) for token in example_df.token]

In [ ]:
example_df.sample(20)

### Lemmatization

Ein Lemma ist die Grundform eines Wortes.  

go, goes, went, gone oder going --> go

In [ ]:
# import von spacy
import spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Process the text with spaCy
doc = nlp(example_text)

In [ ]:
doc[0]

In [ ]:
dir(doc[0])

# https://spacy.io/api/token#attributes

In [ ]:
for i in range(20,50):
    print((doc[i].text, doc[i].lemma_))

In [ ]:
lemma_df = pd.DataFrame({'token': [token.text for token in doc],
                         'lemma': [token.lemma_ for token in doc]})

lemma_df['stem'] = [stemmer.stem(i) for i in lemma_df.token]

In [ ]:
lemma_df.sample(20)

### N-grams

N-grams sind Kombinationen von n Wörtern. gensim kann Worte erkennen, die oft zusammen auftauchen.

In [ ]:
import gensim

In [ ]:
# gensim expect as input tokenized texts
texts = [word_tokenize(text) for text in df.text]

In [ ]:
# extract bigrams
bigrams = gensim.models.Phrases(texts, min_count=5, threshold=100)
texts_bigrams = [bigrams[text] for text in texts]

In [ ]:
# visualize the extracted bigrams
extracted_bigrams = []
for text in texts_bigrams:
    for el in text:
        if "_" in el:
            extracted_bigrams.append(el)

extracted_bigrams = set(extracted_bigrams)
print(extracted_bigrams)

### Stopwords

Stoppwörter sind Wörter, die häufig in einer Sprache verwendet werden, aber normalerweise keine große Bedeutung oder keinen semantischen Wert haben, wenn sie im Kontext verwendet werden. Beispiele für Stoppwörter im Englischen sind "the", "a", "an", "and", "in", "on", "is", "are", "for", "with" und so weiter.

In [ ]:
# Gesamtanzahl der Tokens
total_tokens = sum(vocabulary.values())

# Wörter nach Häufigkeit sortieren
sorted_vocab = vocabulary.most_common(1000)

# Rang (x-Achse) und relative Häufigkeit (y-Achse)
ranks = range(1, len(sorted_vocab) + 1)
relative_freqs = [count / total_tokens for _, count in sorted_vocab]
abs_freqs = [count for _, count in sorted_vocab]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(ranks, abs_freqs)
# plt.xscale('log')
# plt.yscale('log')
plt.xlabel('Rang des Wortes')
plt.ylabel('Häufigkeit')
plt.title('Zipf-Plot der Wortverteilung')
plt.grid(True)
plt.show()

In [ ]:
# Die 20 häufigsten Wörter
top_words = vocabulary.most_common(30)

# X- und Y-Werte vorbereiten
words = [word for word, count in top_words]
counts = [count for word, count in top_words]

# Barplot erzeugen
plt.figure(figsize=(10, 6))
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.xlabel("Wörter")
plt.ylabel("Häufigkeit")
plt.title("Häufigste Wörter")
plt.tight_layout()
plt.show()

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS

In [ ]:
text = df.text[40]

# Process the text with spaCy
doc = nlp(text)

# Define the list of stop words
stop_words = list(STOP_WORDS)

In [ ]:
print(stop_words)

In [ ]:
# Remove stop words from the text
filtered_text = [token.text for token in doc if token.text.lower() not in stop_words]
stop_words_removed = [token.text for token in doc if token.text.lower() in stop_words]

In [ ]:
# Print the original and filtered text, and the stop words removed
print("Original tokens: ", [token.text for token in doc])

In [ ]:
print("Filtered tokens:", filtered_text)

In [ ]:
print("Stop words removed: ", stop_words_removed)

In [ ]:
print(len(stop_words))
stop_words.extend(["bst"])
print(len(stop_words))

In [ ]:
'bst' in stop_words

### Parts of Speech

English has 9 main categories:

verb — Expresses an action or a state of being. E.g. jump, is, write, become  
noun — identifies a person, a place or a thing or names of particular of one of these (pronoun). E.g. man, house, happiness  
pronoun — can replace a noun or noun phrase. E.g. she, we, they, it  
determiner — Is placed in front of a noun to express a quantity or clarify  what the noun refers to 
adjective — modifies a noun or a pronoun. E.g. pretty, old, blue, smart  
adverb — modifies a verb, an adjective, or another adverb. E.g. gently, extremely, carefully, well  
preposition — Connect a noun/pronoun to other parts of the sentence. E.g. by, with, about, until  
conjunction — glue words, clauses, and sentences together. E.g. and, but, or, while, because  
interjection — Expresses emotion in a sudden or exclamatory way. E.g. oh!, wow!, oops!  

In [ ]:
for token in doc:
    print(token.text, token.pos_)

In [ ]:
spacy.explain("PROPN")

In [ ]:
pd.DataFrame({'token': [token.text for token in doc],
             'pos': [token.pos_ for token in doc]}).sample(20)

### Named enitity recognition

In [ ]:
example_text_entities = pd.DataFrame({'entity': [ent.text for ent in doc.ents],
                                      'entity_label': [ent.label_ for ent in doc.ents]})

In [ ]:
example_text_entities.head()

In [ ]:
example_text_entities[example_text_entities.entity_label == 'PERSON'].head()

In [ ]:
example_text_entities[example_text_entities.entity_label == 'PERSON'].value_counts('entity')

### Preprocessing Pipeline

Verschiedene Vorverarbeitungsschritte können wir in einer Pipeline an Funktionen zusammenfassen:

In [ ]:
import spacy
import re # regex
from nltk.tokenize import word_tokenize
from spacy.lang.en.stop_words import STOP_WORDS
nlp = spacy.load("en_core_web_sm")

Sonderzeichen, Zahlen, Zeilenumbrüch etc. entfernen. In dieser Funktion benutzen wir reguläre Ausdrücke `regex`. Eine Übersicht über diese Ausdrücke finden wir z.B. hier: https://images.datacamp.com/image/upload/v1665049611/Marketing/Blog/Regular_Expressions_Cheat_Sheet.pdf

In [ ]:
def clean_text(text):

    # remove punctuation and special characters
    pattern = r"[^\w\s]"
    text_clean = re.sub(pattern, "", text)

    # remove numbers
    pattern = r"\d+"
    text_clean = re.sub(pattern, "", text_clean)

    # remove all non-ASCII characters
    pattern = r"[^\x00-\x7F]+"
    text_clean = re.sub(pattern, "", text_clean)

    # remove new line characters
    text_clean.replace("\n", "")

    # remove empty spaces left by regex
    text_clean = ' '.join(text_clean.split())
    
    return text_clean

In [ ]:
sociology_jokes = [
    "Why don’t sociologists throw wild parties? Too many social constructs.",
    "I told my date I study sociology. They asked, 'So… like, psychology for groups?'",
    "Sociologists do it with structure and agency.",
    "I tried explaining social norms at a party. Turns out, that was breaking one.",
    "Marx walks into a bar. The bartender says, 'You look classless.'"
]

In [ ]:
print(sociology_jokes)

In [ ]:
sociology_jokes_clean = [clean_text(i) for i in sociology_jokes]
sociology_jokes_clean

... Tokenisierung

In [ ]:
def tokenization(texts):
    return [word_tokenize(text) for text in texts]

In [ ]:
sociology_jokes_clean_tokens = tokenization(sociology_jokes_clean)
sociology_jokes_clean_tokens

... Stopwords entfernen

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS

def remove_stop_words(texts, stop_words=[]):
    if stop_words == []:
        stop_words = list(STOP_WORDS)
    return [[word for word in doc if word.lower() not in stop_words] for doc in texts]

In [ ]:
sociology_jokes_clean_tokens_stopwords = remove_stop_words(sociology_jokes_clean_tokens)
sociology_jokes_clean_tokens_stopwords

... Bigrams hinzufügen

In [ ]:
def add_bigrams(texts, min_bigram_count=5):
    bigrams = gensim.models.Phrases(texts, min_count=min_bigram_count, threshold=100)
    return [bigrams[text] for text in texts]

In [ ]:
sociology_jokes_clean_tokens_stopwords_bigrams = add_bigrams(sociology_jokes_clean_tokens_stopwords)
sociology_jokes_clean_tokens_stopwords_bigrams

... stemmen

In [ ]:
def stemming(texts):
    stemmer = PorterStemmer()
    return [[stemmer.stem(word) for word in doc] for doc in texts]

In [ ]:
sociology_jokes_clean_tokens_stopwords_bigrams_stem = stemming(sociology_jokes_clean_tokens_stopwords_bigrams)
sociology_jokes_clean_tokens_stopwords_bigrams_stem

... lemmatisieren

In [ ]:
def lemmatization(texts):
    texts_lemma = []
    for text in texts:
        doc = nlp(" ".join(text)) 
        texts_lemma.append([token.lemma_ for token in doc])
    return texts_lemma

In [ ]:
sociology_jokes_clean_tokens_stopwords_bigrams_lemma = lemmatization(sociology_jokes_clean_tokens_stopwords_bigrams)
sociology_jokes_clean_tokens_stopwords_bigrams_lemma

Alle Funktionen können wir jetzt in eine Pipeline integrieren. Diese Funktion nimmt einen Textkorpus auf. Ein Textkorpus besteht aus einer Reihe an Dokumenten. Diese Dokumente können z.B. in einer Liste oder einem array gespeichert sein.

In [ ]:
stop_words = list(STOP_WORDS)
stop_words.extend(['bst', 'pm', 'gmt', 'update'])

In [ ]:
def pipeline(corpus, custom_stopwords=[]):
    print("Cleaning text...")
    corpus = [clean_text(text) for text in corpus]

    print("Tokenization...")
    corpus = tokenization(corpus)

    print("Lowercasing...")
    corpus = [[el.lower() for el in text] for text in corpus]

    print("Stop Words removal...")
    corpus = remove_stop_words(corpus, stop_words=custom_stopwords)
    
    print("Extract bigrams...")
    corpus = add_bigrams(corpus)

    print("Lemmatization...")
    corpus = lemmatization(corpus)

    print("Stop Words removal after lemmatizing...")
    corpus = remove_stop_words(corpus, stop_words)

    print("Removing tokens that are too short...")
    corpus = [[c for c in text if len(c) > 1] for text in corpus]

    return corpus

In [ ]:
sociology_jokes_preprocessed = pipeline(sociology_jokes, custom_stopwords=stop_words)

In [ ]:
sociology_jokes_preprocessed

Und diese Pipeline können wir nun auf alle möglichen Texte, die in einer Liste array etc gespeichert sind loslassen...

In [ ]:
df.head()

In [ ]:
df.text

In [ ]:
df['text_preprocessed'] = pipeline(df.text)

In [ ]:
df[['text', 'text_preprocessed']].head()

In [ ]:
from gensim.corpora import Dictionary

# we create a dictionary
dictionary = Dictionary(df.text_preprocessed)

In [ ]:
print(f"Number of words in the dictionary: {len(dictionary)}")
print("Dictionary first 5 elements (id, token):", list(dictionary.items())[:5])

In [ ]:
# Absolute Häufigkeit jedes Tokens über alle Dokumente
word_freqs = dictionary.cfs 

# Häufigkeiten sortieren
sorted_freqs = sorted(word_freqs.values(), reverse=True)
ranks = range(1, len(sorted_freqs) + 1)

plt.figure(figsize=(8, 6))
plt.plot(ranks, sorted_freqs)
# plt.xscale('log')
# plt.yscale('log')
plt.xlabel('Wortrang')
plt.ylabel('Dokumentenhäufigkeit')
plt.title('Dokumentenhäufigkeit (vor dem Pruning)')
plt.grid(True)
plt.show()

In [ ]:
# Prune the dictionary: remove words that appear in less than 2 documents
dictionary.filter_extremes(no_below=5, 
                           #no_above=0.95
                          )

# Note: Due to the gap shrinking, the same word may have a different word id before and after the call to this function!

In [ ]:
print(f"Number of words in the dictionary after pruning: {len(dictionary)}")

In [ ]:
# Absolute Häufigkeit jedes Tokens über alle Dokumente
word_freqs = dictionary.cfs 

# Frequenzen sortieren (häufigstes Wort zuerst)
sorted_freqs = sorted(word_freqs.values(), reverse=True)
ranks = range(1, len(sorted_freqs) + 1)

plt.figure(figsize=(8, 6))
plt.plot(ranks, sorted_freqs)
# plt.xscale('log')
# plt.yscale('log')
plt.xlabel('Wortrang')
plt.ylabel('Dokumentenhäufigkeit')
plt.title('Dokumentenhäufigkeit (nach dem Pruning)')
plt.grid(True)
plt.show()

In [ ]:
# covert the corpus to bag of words format 
document_term_matrix = [dictionary.doc2bow(text) for text in df.text_preprocessed]

In [ ]:
print("First document in bag-of-words format (raw):", document_term_matrix[0])

In [ ]:
print("First document in bag-of-words format (word, frequency):",
      [[dictionary[id], freq] for id, freq in document_term_matrix[0]])

Top words im Corpus:

In [ ]:
word_counts_df = pd.DataFrame([[dictionary[id], freq] for id, freq in dictionary.cfs.items()],
                            columns=['word', 'count']).sort_values('count', ascending=False).reset_index(drop=True)

In [ ]:
top_words = word_counts_df.head(25).sort_values(by='count', ascending=True)

# Plot
plt.figure(figsize=(8, 6))
plt.barh(top_words['word'], top_words['count'], color='darkgray')
plt.xlabel('Count')
plt.ylabel('Wort')
plt.title('Die häufigsten Wörter')
plt.tight_layout()
plt.show()

In [ ]:
word_counts_df.tail(20)

### export

In [ ]:
import pickle as pkl

with open("data/dict_gensim.pkl", "wb") as file:
    pkl.dump(dictionary, file)

with open("data/text_df.pkl", "wb") as file:
    pkl.dump(df, file)

with open("data/document_term_matrix.pkl", "wb") as file:
    pkl.dump(document_term_matrix, file)

<div class='alert alert-block alert-success'>

### Aufgabe 1

--> session_05_exercise_01.ipynb

</div>

## Topic Model

Topic Models sind probabilistische Modelle, die zur Bestimmung von semantischen Clustern in Dokumentensammlungen verwendet werden. Sie eignen sich für die Erforschung von Textdaten, da sie thematische Strukturen finden, die nicht im Voraus definiert sind. Die Berechnung zielt darauf ab, die proportionale Zusammensetzung einer festen Anzahl von Themen in den Dokumenten einer Sammlung zu bestimmen. Diese semantischen Cluster können wir als Themen interpretieren.

Topic Modelle liefern Wahrscheinlichkeitsverteilungen über die Menge aller Wörter für jedes Thema und Wahrscheinlichkeitsverteilungen über die Menge der Themen für jedes Dokument. Jede kleinste Analyseeinheit (z. B. ein Wort oder ein n-Gramm) hat eine Wahrscheinlichkeit, zu jedem Thema zu gehören, und jedes Thema hat eine Wahrscheinlichkeit, in jedem Dokument aufzutreten. Ein Thema wird semantisch interpretierbar durch die n wahrscheinlichsten Wörter, die es enthält.

In [ ]:
# import
# Load the gensim dictionary
with open("data/dict_gensim.pkl", "rb") as file:
    dictionary = pkl.load(file)

# Load the DataFrame
with open("data/text_df.pkl", "rb") as file:
    df = pkl.load(file)

# Load the document term matrix
with open("data/document_term_matrix.pkl", "rb") as file:
    document_term_matrix = pkl.load(file)

In [ ]:
import gensim
from gensim.models import LdaMulticore

lda_model = LdaMulticore(
    corpus=document_term_matrix,
    id2word=dictionary,
    num_topics=10,
    passes=10,
    iterations=100,
    workers=3,             # Anzahl der parallelen Prozesse (z. B. 3 Kerne)
    random_state=1234
)

In [ ]:
# print the topics and associated keywords
for topic in lda_model.print_topics(num_topics=10, num_words=10):
    print(topic)

Auswahl des Modells anhand des Kohärenz Scores. 


Die Topic Kohärenz bewerten ein einzelnes Topic, indem sie den Grad der semantischen Ähnlichkeit zwischen hoch bewerteten Wörtern im Thema messen. Diese Messungen helfen bei der Unterscheidung zwischen Themen, die semantisch interpretierbar sind, und Themen, die Artefakte statistischer Inferenz sind.  Zusätzlich können wir verschiedene Modelle mit dem Wert der mittleren Kohärenz vergleichen.

In [ ]:
from gensim.models import LdaMulticore
from gensim.models.coherencemodel import CoherenceModel
import numpy as np

scores = []
models = []

for num_topics in np.arange(5, 30):

    # fit LDA model
    lda_model = LdaMulticore(
        document_term_matrix,
        id2word=dictionary,
        num_topics=num_topics,
        workers=3,             # Anzahl der parallelen Prozesse (z. B. 3 Kerne)
        random_state=12345
    )

    # compute Coherence Score
    coherence_model_lda = CoherenceModel(model=lda_model,
                                         texts=df.text_preprocessed,
                                         dictionary=dictionary)
    
    coherence_lda = coherence_model_lda.get_coherence()
    print(f'Coherence Score with {num_topics} topics: {coherence_lda}')

    scores.append([num_topics, coherence_lda])
    models.append(lda_model)

In [ ]:
scores_df = pd.DataFrame(scores, columns=['num_topic', 'coherence_score'])

In [ ]:
# Plot mit Matplotlib
plt.figure(figsize=(8, 6))
plt.plot(scores_df['num_topic'], scores_df['coherence_score'], marker='o')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score')
plt.title('Coherence Score by Number of Topics')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
scores_df.sort_values('coherence_score', ascending=False)

In [ ]:
# best model
lda_model_best = models[18]

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models

# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model_best,
                                     document_term_matrix,
                                     dictionary)
vis

### Dokumentenhäufigkeiten

In [ ]:
# alle Topic Wahrscheinlichkeiten je Document als tuple
doc_topic_probs_tuples = [i for i in lda_model_best.get_document_topics(document_term_matrix, minimum_probability=0)]
# tuple zu dict für Transformation zu Dataframe
doc_topic_probs_dicts = [dict(doc) for doc in doc_topic_probs_tuples]
# Dataframe der topic Wahrscheinlihckeiten erstellen
doc_topic_probs_df = pd.DataFrame(doc_topic_probs_dicts).add_prefix('topic_')
# Topic Wahrscinelichkeiten an df anfügen
abstracts_topics_df = pd.concat([df, doc_topic_probs_df], axis=1)

In [ ]:
abstracts_topics_df.head()

In [ ]:
abstracts_topics_df.sort_values('topic_3', ascending=False).head(3)

In [ ]:
abstracts_topics_df.sort_values('topic_3', ascending=False).iloc[0].text

<div class='alert alert-block alert-success'>

### Aufgabe 2

--> session_05_exercise_02.ipynb

</div>